# Backend Comparison: ChromaDB vs Qdrant

This notebook compares FFAI's vector store backends side-by-side using
the **factory pattern** (`get_store()`).

- Both backends use in-memory/embedded mode — no servers or API keys
- Same documents, same synthetic embeddings, same queries
- Cleanup removes all temporary data at the end

The factory pattern means you switch backends with a single string:
`get_store(backend='chroma')` vs `get_store(backend='qdrant')`.

<div class="page-break"></div>

---

## Section 1: Setup

In [ ]:
import asyncio
import sys
import tempfile
from pathlib import Path

_cwd = Path().resolve()
_project_root = _cwd
for _p in [_cwd, *list(_cwd.parents)]:
    if (_p / 'pyproject.toml').is_file():
        _project_root = _p
        break

if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))

import numpy as np  # noqa: E402

from ffai.rag.stores import (  # noqa: E402
    get_store,
    list_available_stores,
)

print(f'Available backends: {list_available_stores()}')

<div class="page-break"></div>

---

## Section 2: Prepare Documents and Embeddings

Create a shared set of documents and embeddings so both backends
receive identical data.

In [ ]:
DIM = 128


def make_embedding(text: str, dim: int = DIM) -> list[float]:
    rng = np.random.default_rng(hash(text) & 0xFFFFFFFF)
    vec = rng.standard_normal(dim)
    return (vec / np.linalg.norm(vec)).tolist()


docs = [
    {'id': 'c1', 'text': 'Python asyncio provides asynchronous programming with async/await.',
     'source': 'python.md', 'strategy': 'recursive'},
    {'id': 'c2', 'text': 'Rust ownership ensures memory safety without garbage collection.',
     'source': 'rust.md', 'strategy': 'recursive'},
    {'id': 'c3', 'text': 'Go goroutines are lightweight concurrency primitives.',
     'source': 'go.md', 'strategy': 'recursive'},
    {'id': 'c4', 'text': 'JavaScript promises handle async operations in single-threaded envs.',
     'source': 'js.md', 'strategy': 'markdown'},
    {'id': 'c5', 'text': 'Java threads enable concurrent programming with shared memory.',
     'source': 'java.md', 'strategy': 'markdown'},
]

ids = [d['id'] for d in docs]
texts = [d['text'] for d in docs]
embeddings = [make_embedding(t) for t in texts]
metadatas = [{'source': d['source'], 'chunking_strategy': d['strategy']} for d in docs]

print(f'Prepared {len(docs)} documents with {DIM}-dim embeddings')

<div class="page-break"></div>

---

## Section 3: Create Both Stores

- **ChromaDB**: Persistent client in a temp directory
- **Qdrant**: In-memory mode (`location=":memory:"`)

In [ ]:
chroma_dir = Path(tempfile.mkdtemp(prefix='chroma_compare_'))

store_configs = {
    'chroma': {'collection_name': 'compare_test', 'dir': str(chroma_dir)},
    'qdrant': {'collection_name': 'compare_test', 'embedding_dim': DIM, 'location': ':memory:'},
}

stores = {}
for backend, kwargs in store_configs.items():
    stores[backend] = get_store(backend=backend, **kwargs)
    print(f'{backend}: created (name={stores[backend].name})')

<div class="page-break"></div>

---

## Section 4: Index and Compare

Add the same documents to both backends and verify counts.

In [ ]:
for name, store in stores.items():
    added = asyncio.run(store.aadd(ids, texts, embeddings, metadatas))
    count = store.count()
    sources = store.list_sources()
    print(f'{name}: added={added}, count={count}, sources={sources}')

<div class="page-break"></div>

---

## Section 5: Search and Compare Results

Run the same query against both backends. Because we use synthetic
embeddings derived from text content, the ranking order may differ,
but both backends return results.

In [ ]:
query_text = 'concurrent programming'
query_emb = make_embedding(query_text)

print(f'Query: "{query_text}"\n')
for name, store in stores.items():
    hits = asyncio.run(store.asearch(query_emb, top_k=3))
    print(f'--- {name} ({len(hits)} results) ---')
    for i, h in enumerate(hits, 1):
        print(f'  {i}. [{h.score:.4f}] {h.content[:70]}...')
        print(f'     source={h.source}')
    print()

<div class="page-break"></div>

---

## Section 6: Filtered Search

Filter by `source` metadata on both backends.

In [ ]:
where = {'source': 'python.md'}
print(f'Filter: {where}\n')

for name, store in stores.items():
    hits = asyncio.run(store.asearch(query_emb, top_k=3, where=where))
    print(f'--- {name} ({len(hits)} results) ---')
    for i, h in enumerate(hits, 1):
        print(f'  {i}. [{h.score:.4f}] {h.content[:70]}...')
        print(f'     source={h.source}')
    print()

<div class="page-break"></div>

---

## Section 7: Cleanup

Clear collections and remove temp directories.

In [ ]:
import shutil

for name, store in stores.items():
    store.clear()
    print(f'{name}: cleared (count={store.count()})')

if chroma_dir.exists():
    shutil.rmtree(chroma_dir)
    print(f'\nRemoved ChromaDB dir: {chroma_dir}')

print('\nCleanup complete. No files or data left behind.')